In [ ]:
import osmnx as ox
import pandas as pd
import shapely
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import esda
from libpysal import weights

In [ ]:
G = ox.load_graphml(
    "../data/processed/paris_simplified_results/paris_cleaned_multigraph.graphml"
)
gdf_edges = ox.graph_to_gdfs(G, edges=True, nodes=False)
gdf_edges = gdf_edges[["geometry", "length", "built"]]

In [ ]:
gdf_vote = gpd.read_file("../data/processed/paris_official_data/paris_vote_list.gpkg")
gdf_iris = gpd.read_file(
    "../data/processed/paris_official_data/paris_dem_iris_condensed.gpkg"
)

In [ ]:
gdf_vote = gdf_vote.to_crs(gdf_edges.crs)

In [ ]:
gdf_iris[
    [
        "pop_density",
        "median_income",
        "poverty_rate",
        "commuter_cyclist_share",
        "commuter_driver_share",
    ]
].corr()

In [ ]:
plt.matshow(
    gdf_iris[
        [
            "pop_density",
            "median_income",
            "poverty_rate",
            "commuter_cyclist_share",
            "commuter_driver_share",
        ]
    ].corr(),
    cmap="bwr",
)

In [ ]:
gdf_vote_edges = gdf_vote.sjoin(gdf_edges, how="left", predicate="intersects")

In [ ]:
results_dict = {}
for idx, row in gdf_vote_edges.iterrows():
    if isinstance(row["built"], str):
        length = shapely.intersection(
            row["geometry"], gdf_edges.loc[row["u"], row["v"], row["key"]]["geometry"]
        ).length
    else:
        length = 0
    planned = 0
    before = 0
    built = 0
    if row["built"] == "No":
        planned += length
    elif row["built"] == "2021-01-01":
        before += length
    else:
        planned += length
        built += length
    if idx not in results_dict.keys():
        results_dict[idx] = {
            "length_before_2021": before,
            "length_planned": planned,
            "length_built": built,
        }
    else:
        results_dict[idx]["length_before_2021"] += before
        results_dict[idx]["length_planned"] += planned
        results_dict[idx]["length_built"] += built

In [ ]:
df = pd.DataFrame.from_dict(results_dict, orient="index")

In [ ]:
gdf_vote_res = gdf_vote.merge(df, left_index=True, right_index=True)

In [ ]:
gdf_vote_res["length_accomplished_share"] = (
    gdf_vote_res["length_built"] / gdf_vote_res["length_planned"]
)

In [ ]:
gdf_vote_res["length_final"] = (
    gdf_vote_res["length_before_2021"] + gdf_vote_res["length_planned"]
)

In [ ]:
for col in [
    "length_before_2021",
    "length_planned",
    "length_built",
    "length_final",
]:
    gdf_vote_res[col + "_norm"] = (gdf_vote_res[col] / 10**3) / (
        gdf_vote_res["geometry"].area / 10**6
    )

In [ ]:
gdf_vote_res.plot(column="length_planned_norm", cmap="Oranges")

In [ ]:
gdf_vote_res.plot(column="length_built_norm", cmap="Oranges")

In [ ]:
gdf_vote_res.plot(column="length_before_2021_norm", cmap="Oranges")

In [ ]:
gdf_vote_res.plot(column="length_final_norm", cmap="Oranges")

In [ ]:
gdf_vote_res.plot(column="length_accomplished_share", cmap="Greens")

In [ ]:
for col in [
    "LUG_share",
    "LUC_share",
    "LUD_share",
]:
    mean = gdf_vote_res[col].mean()
    std = gdf_vote_res[col].std()
    gdf_vote_res[col + "_dev"] = (gdf_vote_res[col] - mean) / std

In [ ]:
plt.scatter(gdf_vote_res["LUD_share_dev"], gdf_vote_res["length_before_2021_norm"])

In [ ]:
plt.scatter(gdf_vote_res["LUD_share_dev"], gdf_vote_res["length_planned_norm"])

In [ ]:
plt.scatter(gdf_vote_res["LUD_share_dev"], gdf_vote_res["length_built_norm"])

In [ ]:
plt.scatter(gdf_vote_res["LUD_share_dev"], gdf_vote_res["length_accomplished_share"])

In [ ]:
gdf_vote_res["LUD_share_dev"].values.shape

In [ ]:
gdf_vote_res["length_accomplished_share"].values.shape

In [ ]:
filt = gdf_vote_res[gdf_vote_res["length_accomplished_share"].notna()]

In [ ]:
reg = sklearn.linear_model.LinearRegression().fit(
    filt["LUD_share_dev"].values.reshape(-1, 1),
    filt["length_accomplished_share"].values.reshape(-1, 1),
)

In [ ]:
xx = np.linspace(filt["LUD_share_dev"].min(), filt["LUD_share_dev"].max())
yy = (xx * reg.coef_ + reg.intercept_)[0]

In [ ]:
reg.score(
    filt["LUD_share_dev"].values.reshape(-1, 1),
    filt["length_accomplished_share"].values.reshape(-1, 1),
)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(filt["LUD_share_dev"].values, filt["length_accomplished_share"].values)
ax.plot(xx, yy, color="red")

In [ ]:
reg.coef_

In [ ]:
reg.intercept_

In [ ]:
# Generate W from the GeoDataFrame
# w = weights.KNN.from_dataframe(gdf_vote_res, k=16)
w = weights.Kernel.from_dataframe(gdf_vote_res, function="gaussian", k=5)
# Row-standardization
w.transform = "R"
moran = esda.moran.Moran(gdf_vote_res["length_planned_norm"], w)
moran.I, moran.p_sim

In [ ]:
moran = esda.moran.Moran(gdf_vote_res["length_built_norm"], w)
moran.I, moran.p_sim

In [ ]:
moran = esda.moran.Moran(gdf_vote_res["length_before_2021_norm"], w)
moran.I, moran.p_sim

In [ ]:
w = weights.Kernel.from_dataframe(gdf_iris, function="gaussian", k=5)
w.transform = "R"
moran = esda.moran.Moran(gdf_iris["pop_density"], w)
moran.I, moran.p_sim